# Custom CNN — Four-Breed Dog Classification (PyTorch)

Self-contained notebook for the Custom CNN track.

**Breeds:** French Bulldog, Cardigan, Labrador Retriever, Golden Retriever

**Pipeline:** leaf/image-safe train/val/test split → RGB resize + normalize → train-only mild augmentation → Small CNN from scratch → val tuning → one-shot test evaluation

**How to run (important):**
1. Top-right kernel picker → choose **Python (cnn/.venv)** (not the default `Python 3`).
2. If you see `ModuleNotFoundError: matplotlib` / `torch`, you are on the wrong kernel — switch and re-run.
3. Then **Run All**.

Outputs go to `cnn/outputs/`.


In [1]:
# 1. Setup
import json
import random
import sys
import zipfile
from collections import defaultdict
from pathlib import Path

print("Python executable:", sys.executable)

# Fail fast with a clear message if the wrong Jupyter kernel is selected
try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from PIL import Image
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"{e}\n\nWrong Jupyter kernel. Switch to **Python (cnn/.venv)** "
        f"(kernel name: cnn-venv), then re-run this cell.\n"
        f"Current Python: {sys.executable}"
    ) from e

SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
MAX_EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 32
LR_GRID = [1e-3, 3e-4]
DROPOUT_GRID = [0.3, 0.5]

CLASS_NAMES = ["French_bulldog", "Cardigan", "Labrador_retriever", "golden_retriever"]
BREEDS = {
    "1121-n000002-French_bulldog": ("French_bulldog", 0),
    "2909-n000116-Cardigan": ("Cardigan", 1),
    "3580-n000122-Labrador_retriever": ("Labrador_retriever", 2),
    "5355-n000126-golden_retriever": ("golden_retriever", 3),
}

# Resolve cnn/ even if the notebook cwd is not that folder
_FIXED = Path("/Users/rexfufung/Desktop/AI Group project /cnn").resolve()
_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve() / "cnn",
    _FIXED,
]
CNN_DIR = None
for c in _candidates:
    if (c / "custom_cnn.ipynb").is_file() or (c / "outputs" / "train.csv").is_file():
        CNN_DIR = c
        break
if CNN_DIR is None:
    CNN_DIR = _FIXED

PROJECT_ROOT = CNN_DIR.parent
PAPER_DIR = PROJECT_ROOT / "OneDrive_2026-10-24" / "0. Original Tsinghua Dataset and Paper"
LOW_RES = PAPER_DIR / "low-resolution"
ZIP_PATH = PAPER_DIR / "low-resolution.zip"
OUTPUT_DIR = CNN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()
print(f"CNN_DIR   = {CNN_DIR}")
print(f"LOW_RES   = {LOW_RES}  exists={LOW_RES.is_dir()}")
print(f"OUTPUT_DIR= {OUTPUT_DIR}")
print(f"Device    = {device}")
print(f"Seed      = {SEED}")
print(f"torch     = {torch.__version__}")


Python executable: /Users/rexfufung/Desktop/AI Group project /cnn/.venv/bin/python


Matplotlib is building the font cache; this may take a moment.


CNN_DIR   = /Users/rexfufung/Desktop/AI Group project /cnn
LOW_RES   = /Users/rexfufung/Desktop/AI Group project /OneDrive_2026-10-24/0. Original Tsinghua Dataset and Paper/low-resolution  exists=True
OUTPUT_DIR= /Users/rexfufung/Desktop/AI Group project /cnn/outputs
Device    = mps
Seed      = 42
torch     = 2.14.1


## 2. Data split

Validate RGB images, balance every class to the French Bulldog count, then stratified **70 / 15 / 15** train/val/test.
There are no leaf-group IDs in this Tsinghua low-resolution release, so we split by original image and apply augmentation only in the training loader (never written to disk).

If `train.csv` / `val.csv` / `test.csv` already exist under `outputs/`, this cell reuses them.


In [2]:
# 2. Split (reuse CSVs if present)

def ensure_breed_folders() -> None:
    missing = [folder for folder in BREEDS if not (LOW_RES / folder).is_dir()]
    if not missing:
        print("All breed folders present.")
        return
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Missing {missing} and zip not found: {ZIP_PATH}")
    print(f"Extracting {missing} from {ZIP_PATH.name} ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        members = [
            m for m in zf.namelist()
            if any(m.startswith(f"low-resolution/{folder}/") for folder in missing)
        ]
        zf.extractall(PAPER_DIR, members=members)
    print("Extraction done.")


def collect_valid_images() -> dict[str, list[Path]]:
    by_class: dict[str, list[Path]] = defaultdict(list)
    for folder, (class_name, _) in BREEDS.items():
        breed_dir = LOW_RES / folder
        if not breed_dir.is_dir():
            raise FileNotFoundError(breed_dir)
        paths = sorted(
            p for p in breed_dir.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"} and p.is_file()
        )
        valid = 0
        for path in paths:
            try:
                with Image.open(path) as im:
                    im.convert("RGB")
                by_class[class_name].append(path.resolve())
                valid += 1
            except Exception as exc:
                print(f"  skip {path.name}: {exc}")
        print(f"{class_name}: {valid}/{len(paths)} readable")
    return dict(by_class)


def balance_and_split(by_class: dict[str, list[Path]], seed: int = SEED):
    rng = random.Random(seed)
    min_count = min(len(v) for v in by_class.values())
    print(f"Balancing all classes to {min_count} images (seed={seed})")

    rows_by_split = {"train": [], "val": [], "test": []}
    class_counts = {"train": {}, "val": {}, "test": {}, "balanced_per_class": min_count}

    for _, (class_name, label) in BREEDS.items():
        paths = list(by_class[class_name])
        rng.shuffle(paths)
        paths = paths[:min_count]
        n_train = int(round(min_count * TRAIN_RATIO))
        n_val = int(round(min_count * VAL_RATIO))
        splits = {
            "train": paths[:n_train],
            "val": paths[n_train : n_train + n_val],
            "test": paths[n_train + n_val :],
        }
        for split_name, split_paths in splits.items():
            class_counts[split_name][class_name] = len(split_paths)
            for p in split_paths:
                rows_by_split[split_name].append(
                    {"path": str(p), "label": label, "class_name": class_name}
                )

    for split_name in rows_by_split:
        rng.shuffle(rows_by_split[split_name])

    meta = {
        "seed": seed,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "breeds": {
            folder: {"class_name": name, "label": label}
            for folder, (name, label) in BREEDS.items()
        },
        "class_counts": class_counts,
        "note": (
            "No leaf-group IDs in Tsinghua low-resolution set; each JPEG is one "
            "original image. Split is by image; augmentation is applied only in "
            "the training DataLoader and never written to disk."
        ),
    }
    return rows_by_split["train"], rows_by_split["val"], rows_by_split["test"], meta


train_csv = OUTPUT_DIR / "train.csv"
val_csv = OUTPUT_DIR / "val.csv"
test_csv = OUTPUT_DIR / "test.csv"
meta_path = OUTPUT_DIR / "split_meta.json"

if train_csv.is_file() and val_csv.is_file() and test_csv.is_file():
    print("Reusing existing split CSVs.")
    if meta_path.is_file():
        with meta_path.open(encoding="utf-8") as f:
            split_meta = json.load(f)
    else:
        split_meta = {"seed": SEED}
else:
    ensure_breed_folders()
    by_class = collect_valid_images()
    train_rows, val_rows, test_rows, split_meta = balance_and_split(by_class)
    for name, rows in (("train", train_rows), ("val", val_rows), ("test", test_rows)):
        pd.DataFrame(rows).to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
        print(f"Wrote {name}.csv ({len(rows)} rows)")
    # preserve mean/std if already present
    if meta_path.is_file():
        with meta_path.open(encoding="utf-8") as f:
            old = json.load(f)
        for k in ("mean", "std"):
            if k in old:
                split_meta[k] = old[k]
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(split_meta, f, indent=2)

print(f"Seed: {split_meta.get('seed', SEED)}")
print("Class counts:")
print(json.dumps(split_meta.get("class_counts", {}), indent=2))
for name in ("train", "val", "test"):
    df = pd.read_csv(OUTPUT_DIR / f"{name}.csv")
    print(f"{name}: {len(df)} images | labels={sorted(df['label'].unique().tolist())}")


Reusing existing split CSVs.
Seed: 42
Class counts:
{
  "train": {
    "French_bulldog": 785,
    "Cardigan": 785,
    "Labrador_retriever": 785,
    "golden_retriever": 785
  },
  "val": {
    "French_bulldog": 168,
    "Cardigan": 168,
    "Labrador_retriever": 168,
    "golden_retriever": 168
  },
  "test": {
    "French_bulldog": 168,
    "Cardigan": 168,
    "Labrador_retriever": 168,
    "golden_retriever": 168
  },
  "balanced_per_class": 1121
}
train: 3140 images | labels=[0, 1, 2, 3]
val: 672 images | labels=[0, 1, 2, 3]
test: 672 images | labels=[0, 1, 2, 3]


## 3. Image preprocessing

Shared for all splits: RGB → resize 224 → train-set mean/std normalize.  
**Train only:** resize 256 → random crop 224, horizontal flip, ±15° rotation, light ColorJitter (breed-cue preserving).  
Val / test: deterministic resize + the same normalization (no random transforms).


In [3]:
# 3. Dataset, transforms, loaders

class DogBreedDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def compute_train_mean_std(csv_path, image_size=224, max_samples=512):
    to_tensor = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    ds = DogBreedDataset(csv_path, transform=to_tensor)
    n = min(len(ds), max_samples)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    sum_ = torch.zeros(3)
    sum_sq = torch.zeros(3)
    count = 0
    for images, _ in loader:
        b = images.size(0)
        flat = images.view(b, 3, -1)
        sum_ += flat.sum(dim=(0, 2))
        sum_sq += (flat ** 2).sum(dim=(0, 2))
        count += b * flat.size(2)
        if count // (image_size * image_size) >= n:
            break
    mean = (sum_ / count).tolist()
    var = sum_sq / count - torch.tensor(mean) ** 2
    std = torch.sqrt(var.clamp(min=1e-8)).tolist()
    return mean, std


def build_transforms(mean, std, train: bool):
    normalize = transforms.Normalize(mean=mean, std=std)
    if train:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.0),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize,
    ])


# Load or compute normalization stats from training set only
if meta_path.is_file():
    with meta_path.open(encoding="utf-8") as f:
        split_meta = json.load(f)
else:
    split_meta = {}

if "mean" in split_meta and "std" in split_meta:
    mean, std = split_meta["mean"], split_meta["std"]
    print("Using saved train mean/std from split_meta.json")
else:
    mean, std = compute_train_mean_std(train_csv)
    split_meta["mean"] = mean
    split_meta["std"] = std
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(split_meta, f, indent=2)
    print("Computed and saved train mean/std")

print(f"mean={mean}")
print(f"std ={std}")

train_ds = DogBreedDataset(train_csv, transform=build_transforms(mean, std, train=True))
val_ds = DogBreedDataset(val_csv, transform=build_transforms(mean, std, train=False))
test_ds = DogBreedDataset(test_csv, transform=build_transforms(mean, std, train=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Quick label / shape check
xb, yb = next(iter(train_loader))
print(f"batch images: {tuple(xb.shape)}  labels: {yb[:8].tolist()}")
assert xb.shape[1:] == (3, 224, 224)
assert set(int(x) for x in yb.unique().tolist()).issubset({0, 1, 2, 3})
print("Preprocessing + label check OK")


Using saved train mean/std from split_meta.json
mean=[0.5296207070350647, 0.4922817051410675, 0.44506850838661194]
std =[0.2501884698867798, 0.24680933356285095, 0.25292617082595825]
batch images: (32, 3, 224, 224)  labels: [3, 3, 2, 0, 0, 0, 3, 0]
Preprocessing + label check OK


## 4. Small CNN (from scratch)

Four blocks: `Conv3×3 → BatchNorm → ReLU → MaxPool2`, channels 32 → 64 → 128 → 256, then global average pool, dropout, linear → 4 classes. No pretrained weights.


In [4]:
# 4. Model

class SmallCNN(nn.Module):
    """Four Conv-BN-ReLU-MaxPool blocks (32→64→128→256), GAP, dropout, linear."""

    def __init__(self, num_classes: int = 4, dropout: float = 0.3):
        super().__init__()
        channels = [32, 64, 128, 256]
        layers = []
        in_ch = 3
        for out_ch in channels:
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ])
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels[-1], num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


# Sanity check
_m = SmallCNN().to(device)
with torch.no_grad():
    out = _m(xb.to(device))
print(f"SmallCNN params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"logits shape: {tuple(out.shape)}")
del _m


SmallCNN params: 389,924
logits shape: (32, 4)


## 5. Train + validation tuning

Tune on **validation only**: lr ∈ {1e-3, 3e-4}, dropout ∈ {0.3, 0.5}.  
Early stop on validation macro-F1 (patience 5, max 20 epochs).  
Keep the checkpoint with the best validation macro-F1.


In [5]:
# 5. Train / tune

@torch.no_grad()
def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    avg_loss = total_loss / max(n, 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))
    return avg_loss, acc, macro_f1, cm


def train_one_run(train_loader, val_loader, device, lr, dropout):
    model = SmallCNN(num_classes=len(CLASS_NAMES), dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [], "val_macro_f1": [],
    }
    best_f1, best_state, best_epoch = -1.0, None, 0
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / max(total, 1)
        train_acc = correct / max(total, 1)
        val_loss, val_acc, val_f1, _ = evaluate(model, val_loader, device, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_f1)

        print(
            f"  epoch {epoch:02d}/{MAX_EPOCHS}  "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"  early stop at epoch {epoch} (best val macro-F1={best_f1:.4f} @ {best_epoch})")
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    return {
        "model": model,
        "history": history,
        "best_val_macro_f1": best_f1,
        "best_epoch": best_epoch,
        "lr": lr,
        "dropout": dropout,
    }


tuning_results = []
best_run = None

for lr in LR_GRID:
    for dropout in DROPOUT_GRID:
        print(f"\n=== Tuning lr={lr}, dropout={dropout} ===")
        set_seed(SEED)
        run = train_one_run(train_loader, val_loader, device, lr=lr, dropout=dropout)
        summary = {
            "lr": lr,
            "dropout": dropout,
            "best_val_macro_f1": run["best_val_macro_f1"],
            "best_epoch": run["best_epoch"],
        }
        tuning_results.append(summary)
        print(f"  best val macro-F1={run['best_val_macro_f1']:.4f} @ epoch {run['best_epoch']}")
        if best_run is None or run["best_val_macro_f1"] > best_run["best_val_macro_f1"]:
            best_run = run

print(
    f"\nSelected config: lr={best_run['lr']}, dropout={best_run['dropout']}, "
    f"val_macro_f1={best_run['best_val_macro_f1']:.4f}"
)

ckpt_path = OUTPUT_DIR / "best_small_cnn.pt"
torch.save(
    {
        "state_dict": best_run["model"].state_dict(),
        "lr": best_run["lr"],
        "dropout": best_run["dropout"],
        "best_val_macro_f1": best_run["best_val_macro_f1"],
        "best_epoch": best_run["best_epoch"],
        "class_names": CLASS_NAMES,
        "seed": SEED,
    },
    ckpt_path,
)
print(f"Saved checkpoint → {ckpt_path}")



=== Tuning lr=0.001, dropout=0.3 ===


  epoch 01/20  train_loss=1.3460 train_acc=0.3392  val_loss=1.3737 val_acc=0.3393 val_macro_f1=0.2957


  epoch 02/20  train_loss=1.3101 train_acc=0.3675  val_loss=1.3417 val_acc=0.3557 val_macro_f1=0.2925


  epoch 03/20  train_loss=1.3055 train_acc=0.3783  val_loss=1.7574 val_acc=0.2530 val_macro_f1=0.1065


  epoch 04/20  train_loss=1.2917 train_acc=0.3939  val_loss=1.3590 val_acc=0.3631 val_macro_f1=0.3109


  epoch 05/20  train_loss=1.2825 train_acc=0.3920  val_loss=1.2988 val_acc=0.3601 val_macro_f1=0.3164


... (68 outputs left out)

  epoch 17/20  train_loss=1.0618 train_acc=0.5503  val_loss=1.1921 val_acc=0.4449 val_macro_f1=0.4388


  epoch 18/20  train_loss=1.0626 train_acc=0.5484  val_loss=1.3872 val_acc=0.4196 val_macro_f1=0.3672


  epoch 19/20  train_loss=1.0548 train_acc=0.5618  val_loss=1.2027 val_acc=0.4390 val_macro_f1=0.4102


  epoch 20/20  train_loss=1.0301 train_acc=0.5561  val_loss=1.1563 val_acc=0.4628 val_macro_f1=0.4614
  early stop at epoch 20 (best val macro-F1=0.4803 @ 15)
  best val macro-F1=0.4803 @ epoch 15

Selected config: lr=0.001, dropout=0.3, val_macro_f1=0.5101
Saved checkpoint → /Users/rexfufung/Desktop/AI Group project /cnn/outputs/best_small_cnn.pt


## 6. Curves + one-shot test evaluation

Validation was used only for tuning. The test set is evaluated **once** with the selected checkpoint.


In [6]:
# 6. Plots + final test

def plot_curves(history, out_path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(epochs, history["train_acc"], label="train acc")
    axes[1].plot(epochs, history["val_acc"], label="val acc")
    axes[1].plot(epochs, history["val_macro_f1"], label="val macro-F1")
    axes[1].set_title("Accuracy / Macro-F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


def plot_confusion(cm, out_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Test Confusion Matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


plot_curves(best_run["history"], OUTPUT_DIR / "training_curves.png")

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, test_f1, cm = evaluate(best_run["model"], test_loader, device, criterion)
plot_confusion(cm, OUTPUT_DIR / "confusion_matrix.png")

metrics = {
    "seed": SEED,
    "device": str(device),
    "selected_lr": best_run["lr"],
    "selected_dropout": best_run["dropout"],
    "best_val_macro_f1": best_run["best_val_macro_f1"],
    "best_epoch": best_run["best_epoch"],
    "tuning_results": tuning_results,
    "test": {
        "loss": float(test_loss),
        "accuracy": float(test_acc),
        "macro_f1": float(test_f1),
        "confusion_matrix": cm.tolist(),
        "class_names": CLASS_NAMES,
    },
    "class_counts": split_meta.get("class_counts"),
    "mean": mean,
    "std": std,
}
metrics_path = OUTPUT_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("=== Final test (one-shot) ===")
print(f"seed={SEED}")
print(f"accuracy={test_acc:.4f}  macro_f1={test_f1:.4f}  loss={test_loss:.4f}")
print("confusion matrix:")
print(cm)
print(f"Wrote {metrics_path}")
print(f"Wrote {OUTPUT_DIR / 'training_curves.png'}")
print(f"Wrote {OUTPUT_DIR / 'confusion_matrix.png'}")


/var/folders/tx/385ppbgs00jbh4n3ndxtl9ch0000gn/T/ipykernel_87395/3032820080.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== Final test (one-shot) ===
seed=42
accuracy=0.5506  macro_f1=0.5533  loss=1.0720
confusion matrix:
[[ 78  24  62   4]
 [ 11  99  31  27]
 [ 49  26  80  13]
 [  7  23  25 113]]
Wrote /Users/rexfufung/Desktop/AI Group project /cnn/outputs/metrics.json
Wrote /Users/rexfufung/Desktop/AI Group project /cnn/outputs/training_curves.png
Wrote /Users/rexfufung/Desktop/AI Group project /cnn/outputs/confusion_matrix.png


/var/folders/tx/385ppbgs00jbh4n3ndxtl9ch0000gn/T/ipykernel_87395/3032820080.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Predict a new image

Load `outputs/best_small_cnn.pt` and classify one photo.

**How to use**
1. Put your dog photos into `cnn/input_dog/` (`.jpg` / `.jpeg` / `.png`).
2. Run the next cell.
3. Optional: set `IMAGE_PATH` to a specific file; leave `None` to auto-pick the first image in `input_dog/` (if empty, uses a random test image).

Examples:
```python
IMAGE_PATH = None  # auto from input_dog/
IMAGE_PATH = "input_dog/my_dog.jpg"
```

In [ ]:
# 7. Predict one image (upload = set IMAGE_PATH to your file)

from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

# >>> 改呢度：填相路徑；留 None = 自動用 input_dog/ 入面第一張相
IMAGE_PATH = None  # e.g. "input_dog/my_dog.jpg"

# Resolve paths even if earlier cells were not run
_FIXED = Path("/Users/rexfufung/Desktop/AI Group project /cnn").resolve()
CNN_DIR = _FIXED if (_FIXED / "outputs" / "best_small_cnn.pt").is_file() else Path.cwd().resolve()
if CNN_DIR.name != "cnn" and (CNN_DIR / "cnn" / "outputs" / "best_small_cnn.pt").is_file():
    CNN_DIR = CNN_DIR / "cnn"
OUTPUT_DIR = CNN_DIR / "outputs"
INPUT_DIR = CNN_DIR / "input_dog"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUTPUT_DIR / "best_small_cnn.pt"
META_PATH = OUTPUT_DIR / "split_meta.json"
TEST_CSV = OUTPUT_DIR / "test.csv"

if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}\nPlease train first (section 5) or keep outputs/best_small_cnn.pt.")

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
CLASS_NAMES = ckpt.get("class_names") or [
    "French_bulldog", "Cardigan", "Labrador_retriever", "golden_retriever"
]
dropout = float(ckpt.get("dropout", 0.3))

with META_PATH.open(encoding="utf-8") as f:
    meta = json.load(f)
mean, std = meta["mean"], meta["std"]

# Same eval preprocessing as val/test (no random augmentation)
predict_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 4, dropout: float = 0.3):
        super().__init__()
        channels = [32, 64, 128, 256]
        layers = []
        in_ch = 3
        for out_ch in channels:
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ])
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels[-1], num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = SmallCNN(num_classes=len(CLASS_NAMES), dropout=dropout).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# Pick image: explicit path → first file in input_dog/ → random test demo
def _list_input_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    )

true_name = None
if IMAGE_PATH is not None:
    image_path = Path(IMAGE_PATH).expanduser()
    if not image_path.is_absolute():
        image_path = (CNN_DIR / image_path).resolve()
    else:
        image_path = image_path.resolve()
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
elif _list_input_images(INPUT_DIR):
    image_path = _list_input_images(INPUT_DIR)[0]
    print(f"Using first image in input_dog/: {image_path.name}")
    print(f"(folder has {len(_list_input_images(INPUT_DIR))} image(s); set IMAGE_PATH to pick another)")
else:
    test_df = pd.read_csv(TEST_CSV)
    row = test_df.sample(1, random_state=random.randint(0, 10_000)).iloc[0]
    image_path = Path(row["path"])
    true_name = row["class_name"]
    print(f"input_dog/ is empty — demo with random test image")
    print(f"  path = {image_path}")
    print(f"  true  = {true_name}")
    print(f"Put your photos in: {INPUT_DIR}")

img = Image.open(image_path).convert("RGB")
tensor = predict_tf(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1)[0].cpu()

pred_idx = int(probs.argmax().item())
pred_name = CLASS_NAMES[pred_idx]
pred_prob = float(probs[pred_idx].item())

print("\n=== Prediction ===")
print(f"image     : {image_path.name}")
print(f"predicted : {pred_name}  ({pred_prob*100:.1f}%)")
if true_name is not None:
    print(f"true label: {true_name}  ({'OK' if true_name == pred_name else 'WRONG'})")
print("\nAll class probabilities:")
for name, p in sorted(zip(CLASS_NAMES, probs.tolist()), key=lambda x: -x[1]):
    print(f"  {name:22s} {p*100:5.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img)
axes[0].axis("off")
title = f"Pred: {pred_name} ({pred_prob*100:.1f}%)"
if true_name is not None:
    title += f"\nTrue: {true_name}"
axes[0].set_title(title)

order = sorted(range(len(CLASS_NAMES)), key=lambda i: -probs[i].item())
axes[1].barh([CLASS_NAMES[i] for i in order][::-1], [probs[i].item() for i in order][::-1], color="steelblue")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Probability")
axes[1].set_title("Class probabilities")
fig.tight_layout()
plt.show()